# 🔍 Détection de Fraude dans les Notes de Frais et Remboursements

**Thème :** Audit financier & Détection d'anomalies  
**Type de tâche :** Classification binaire  
**Variable cible :** Note de frais frauduleuse : `oui / non`  
**Source de données :** Expense Fraud Detection Dataset (Kaggle) + données simulées RH  

---

**Intérêt pédagogique / métier :**  
- Audit interne des dépenses  
- Ingénierie de features temporelles et géographiques  
- Seuils d'alerte automatiques  

---

## 📦 0. Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE

# Affichage
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ Bibliothèques importées avec succès.")

---
## 📂 1. Chargement et simulation des données

> **Source :** Expense Fraud Detection Dataset (Kaggle) enrichi par des données RH simulées.
> Si le dataset Kaggle est disponible localement, remplacez le bloc de simulation par `pd.read_csv('chemin/vers/fichier.csv')`.

In [ ]:
np.random.seed(42)
n = 5000

categories = ['Transport', 'Hébergement', 'Repas', 'Fournitures', 'Téléphone', 'Formation']
departements = ['Ventes', 'IT', 'RH', 'Finance', 'Marketing', 'Direction']
villes_fr = ['Paris', 'Lyon', 'Marseille', 'Toulouse', 'Bordeaux']
villes_etrangeres = ['Londres', 'Dubai', 'New York', 'Singapour', 'Zurich']

# --- Variables de base ---
df = pd.DataFrame({
    'employee_id'      : np.random.randint(1000, 1200, n),
    'date_soumission'  : pd.date_range('2022-01-01', periods=n, freq='6H'),
    'montant'          : np.round(np.random.exponential(scale=150, size=n), 2),
    'categorie'        : np.random.choice(categories, n, p=[0.25, 0.20, 0.30, 0.10, 0.10, 0.05]),
    'departement'      : np.random.choice(departements, n),
    'ville'            : np.random.choice(villes_fr + villes_etrangeres, n,
                                          p=[0.25, 0.20, 0.15, 0.15, 0.10, 0.05, 0.04, 0.03, 0.02, 0.01]),
    'nb_justificatifs' : np.random.randint(0, 4, n),
    'anciennete_ans'   : np.random.randint(0, 20, n),
    'salaire_mensuel'  : np.random.randint(2500, 8000, n),
    'manager_approval' : np.random.choice([0, 1], n, p=[0.15, 0.85]),
})

# --- Cible frauduleuse : règles métier + bruit ---
score_fraude = (
    (df['montant'] > 500).astype(int) * 2 +
    (df['nb_justificatifs'] == 0).astype(int) * 3 +
    (df['manager_approval'] == 0).astype(int) * 2 +
    (df['ville'].isin(villes_etrangeres)).astype(int) * 1 +
    np.random.binomial(1, 0.05, n)
)
df['fraude'] = (score_fraude >= 5).astype(int)

print(f"✅ Dataset simulé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"   Taux de fraude : {df['fraude'].mean():.2%}")
df.head()

---
## 🔎 2. Exploration et analyse descriptive (EDA)

In [ ]:
print("=" * 50)
print("📊 Informations générales")
print("=" * 50)
df.info()
print("\n📈 Statistiques descriptives")
df.describe()

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['fraude'].value_counts()
labels = ['Légitimes', 'Frauduleuses']
colors = ['#2196F3', '#F44336']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution des notes de frais', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Répartition (%)', fontsize=13, fontweight='bold')

plt.suptitle('⚠️ Déséquilibre des classes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution du montant par classe
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, ax in zip([0, 1], ['#2196F3', '#F44336'], axes):
    subset = df[df['fraude'] == label]['montant']
    ax.hist(subset, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f"Montant — {'Fraude ❌' if label else 'Légitime ✅'}", fontweight='bold')
    ax.set_xlabel('Montant (€)')
    ax.set_ylabel('Fréquence')
    ax.axvline(subset.median(), color='black', linestyle='--', label=f'Médiane: {subset.median():.0f}€')
    ax.legend()

plt.suptitle('Distribution des montants selon la classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Taux de fraude par catégorie et département
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

taux_cat = df.groupby('categorie')['fraude'].mean().sort_values(ascending=False)
taux_dep = df.groupby('departement')['fraude'].mean().sort_values(ascending=False)

taux_cat.plot(kind='bar', ax=axes[0], color='#FF7043', edgecolor='white')
axes[0].set_title('Taux de fraude par catégorie', fontweight='bold')
axes[0].set_ylabel('Taux de fraude')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

taux_dep.plot(kind='bar', ax=axes[1], color='#AB47BC', edgecolor='white')
axes[1].set_title('Taux de fraude par département', fontweight='bold')
axes[1].set_ylabel('Taux de fraude')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.suptitle('Analyse du taux de fraude par groupe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚙️ 3. Ingénierie des features

On crée des variables **temporelles**, **géographiques** et **comportementales** pour enrichir le modèle.

In [ ]:
df_feat = df.copy()

# --- Features temporelles ---
df_feat['heure_soumission']      = df_feat['date_soumission'].dt.hour
df_feat['jour_semaine']          = df_feat['date_soumission'].dt.dayofweek   # 0=Lundi
df_feat['est_weekend']           = (df_feat['jour_semaine'] >= 5).astype(int)
df_feat['est_hors_heures']       = ((df_feat['heure_soumission'] < 8) |
                                    (df_feat['heure_soumission'] > 19)).astype(int)
df_feat['mois']                  = df_feat['date_soumission'].dt.month
df_feat['fin_trimestre']         = df_feat['mois'].isin([3, 6, 9, 12]).astype(int)

# --- Features géographiques ---
villes_etrangeres = ['Londres', 'Dubai', 'New York', 'Singapour', 'Zurich']
df_feat['est_etranger']          = df_feat['ville'].isin(villes_etrangeres).astype(int)

# --- Features comportementales & financières ---
df_feat['ratio_montant_salaire'] = df_feat['montant'] / df_feat['salaire_mensuel']
df_feat['montant_eleve']         = (df_feat['montant'] > df_feat['montant'].quantile(0.90)).astype(int)
df_feat['sans_justificatif']     = (df_feat['nb_justificatifs'] == 0).astype(int)

# --- Historique par employé ---
hist = df_feat.groupby('employee_id').agg(
    freq_soumissions=('montant', 'count'),
    montant_moyen_emp=('montant', 'mean'),
    montant_max_emp=('montant', 'max')
).reset_index()
df_feat = df_feat.merge(hist, on='employee_id', how='left')

# Z-score du montant par rapport à l'historique de l'employé
df_feat['zscore_montant_emp'] = (
    (df_feat['montant'] - df_feat['montant_moyen_emp']) /
    (df_feat.groupby('employee_id')['montant'].transform('std').fillna(1))
)

print(f"✅ Features créées. Nouveau shape : {df_feat.shape}")
df_feat[['montant', 'heure_soumission', 'est_weekend', 'est_etranger',
          'ratio_montant_salaire', 'zscore_montant_emp', 'fraude']].head()

In [ ]:
# Corrélation des nouvelles features avec la cible
num_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
corr_target = df_feat[num_cols].corr()['fraude'].drop('fraude').sort_values()

plt.figure(figsize=(10, 6))
colors_corr = ['#F44336' if x > 0 else '#2196F3' for x in corr_target]
corr_target.plot(kind='barh', color=colors_corr)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Corrélation des features avec la variable cible (fraude)', fontweight='bold', fontsize=13)
plt.xlabel('Corrélation de Pearson')
plt.tight_layout()
plt.show()

---
## 🧹 4. Préparation des données

In [ ]:
# Encodage des variables catégorielles
df_model = df_feat.copy()
le = LabelEncoder()

for col in ['categorie', 'departement', 'ville']:
    df_model[col + '_enc'] = le.fit_transform(df_model[col])

# Sélection des features
feature_cols = [
    'montant', 'nb_justificatifs', 'anciennete_ans', 'salaire_mensuel',
    'manager_approval', 'heure_soumission', 'jour_semaine', 'mois',
    'est_weekend', 'est_hors_heures', 'fin_trimestre', 'est_etranger',
    'ratio_montant_salaire', 'montant_eleve', 'sans_justificatif',
    'freq_soumissions', 'montant_moyen_emp', 'montant_max_emp',
    'zscore_montant_emp', 'categorie_enc', 'departement_enc', 'ville_enc'
]

X = df_model[feature_cols]
y = df_model['fraude']

# Split train/test stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Rééchantillonnage avec SMOTE (déséquilibre des classes)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Normalisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

print(f"✅ Train (après SMOTE) : {X_train_res.shape} | Test : {X_test.shape}")
print(f"   Distribution y_train_res : {pd.Series(y_train_res).value_counts().to_dict()}")

---
## 🤖 5. Entraînement des modèles

In [ ]:
models = {
    'Régression Logistique' : LogisticRegression(max_iter=1000, random_state=42),
    'Arbre de Décision'     : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest'         : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                      random_state=42, n_jobs=-1),
    'Gradient Boosting'     : GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                                          max_depth=4, random_state=42)
}

resultats = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Modèle':<28} {'ROC-AUC CV':>12} {'±':>6}")
print("-" * 50)

for nom, model in models.items():
    # Utilisation des données scalées pour LR, brutes pour les arbres
    X_tr = X_train_sc if nom == 'Régression Logistique' else X_train_res
    X_te = X_test_sc  if nom == 'Régression Logistique' else X_test

    scores = cross_val_score(model, X_tr, y_train_res, cv=cv, scoring='roc_auc')
    model.fit(X_tr, y_train_res)
    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred  = model.predict(X_te)

    resultats[nom] = {
        'model': model,
        'y_proba': y_proba,
        'y_pred': y_pred,
        'auc_cv': scores.mean(),
        'auc_cv_std': scores.std(),
        'auc_test': roc_auc_score(y_test, y_proba),
        'X_te': X_te
    }
    print(f"{nom:<28} {scores.mean():>12.4f} {scores.std():>6.4f}")

print("\n✅ Entraînement terminé.")

---
## 📊 6. Évaluation des modèles

In [ ]:
# Courbes ROC de tous les modèles
plt.figure(figsize=(9, 6))
colors_roc = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for (nom, res), color in zip(resultats.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, lw=2, color=color,
             label=f"{nom} (AUC = {res['auc_test']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
plt.title('Courbes ROC — Comparaison des modèles', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rapport de classification du meilleur modèle
best_nom = max(resultats, key=lambda k: resultats[k]['auc_test'])
best_res = resultats[best_nom]

print(f"🏆 Meilleur modèle : {best_nom}")
print(f"   AUC Test : {best_res['auc_test']:.4f}")
print("\n" + "="*55)
print("📋 Rapport de classification")
print("="*55)
print(classification_report(y_test, best_res['y_pred'],
                             target_names=['Légitime', 'Fraude']))

In [ ]:
# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best_res['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Légitime', 'Fraude'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Matrice de confusion — {best_nom}', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Courbe Précision-Rappel
plt.figure(figsize=(8, 5))
precision, recall, thresholds = precision_recall_curve(y_test, best_res['y_proba'])
plt.plot(recall, precision, color='#E91E63', lw=2)
plt.fill_between(recall, precision, alpha=0.15, color='#E91E63')
plt.xlabel('Rappel (Recall)', fontsize=11)
plt.ylabel('Précision', fontsize=11)
plt.title(f'Courbe Précision-Rappel — {best_nom}', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔑 7. Importance des features

In [ ]:
# Extraction importance (pour les modèles à base d'arbres)
best_model = best_res['model']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=True).tail(15)

    plt.figure(figsize=(10, 6))
    bars = plt.barh(importances.index, importances.values,
                    color=plt.cm.RdYlGn(importances.values / importances.values.max()))
    plt.title(f'Top 15 features importantes — {best_nom}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    # Coefficients pour la régression logistique
    coef = pd.Series(best_model.coef_[0], index=feature_cols).sort_values()
    plt.figure(figsize=(10, 6))
    colors_coef = ['#F44336' if x > 0 else '#2196F3' for x in coef]
    coef.plot(kind='barh', color=colors_coef)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('Coefficients — Régression Logistique', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 🚨 8. Système d'alertes automatiques par seuils

On définit des **niveaux de risque** basés sur le score de probabilité de fraude.

In [ ]:
# Seuils d'alerte
SEUIL_CRITIQUE = 0.75
SEUIL_ELEVE    = 0.50
SEUIL_MODERE   = 0.25

def niveau_risque(proba):
    if proba >= SEUIL_CRITIQUE:
        return '🔴 CRITIQUE'
    elif proba >= SEUIL_ELEVE:
        return '🟠 ÉLEVÉ'
    elif proba >= SEUIL_MODERE:
        return '🟡 MODÉRÉ'
    else:
        return '🟢 FAIBLE'

# Tableau de bord des alertes sur le jeu de test
df_alertes = X_test.copy()
df_alertes['proba_fraude']  = best_res['y_proba']
df_alertes['pred_fraude']   = best_res['y_pred']
df_alertes['reel_fraude']   = y_test.values
df_alertes['niveau_risque'] = df_alertes['proba_fraude'].apply(niveau_risque)

# Résumé par niveau de risque
resume = df_alertes['niveau_risque'].value_counts()
print("📊 Résumé des alertes sur le jeu de test :")
print(resume.to_string())

print("\n🔴 Top 10 notes de frais les plus suspectes :")
top_suspects = df_alertes.nlargest(10, 'proba_fraude')[[
    'montant', 'nb_justificatifs', 'manager_approval',
    'est_etranger', 'proba_fraude', 'niveau_risque', 'reel_fraude'
]]
top_suspects.index = range(1, 11)
top_suspects

In [ ]:
# Visualisation des seuils
plt.figure(figsize=(11, 5))

probas_sorted = np.sort(best_res['y_proba'])[::-1]
plt.plot(probas_sorted, color='#607D8B', lw=1.5, label='Score de fraude')
plt.axhline(SEUIL_CRITIQUE, color='#F44336', lw=2, linestyle='--', label=f'Seuil critique ({SEUIL_CRITIQUE})')
plt.axhline(SEUIL_ELEVE,    color='#FF9800', lw=2, linestyle='--', label=f'Seuil élevé ({SEUIL_ELEVE})')
plt.axhline(SEUIL_MODERE,   color='#FFEB3B', lw=2, linestyle='--', label=f'Seuil modéré ({SEUIL_MODERE})')

plt.fill_between(range(len(probas_sorted)), SEUIL_CRITIQUE, probas_sorted,
                 where=probas_sorted >= SEUIL_CRITIQUE, alpha=0.2, color='#F44336')
plt.fill_between(range(len(probas_sorted)), SEUIL_ELEVE, probas_sorted,
                 where=(probas_sorted >= SEUIL_ELEVE) & (probas_sorted < SEUIL_CRITIQUE),
                 alpha=0.2, color='#FF9800')

plt.xlabel('Notes de frais (triées par risque décroissant)', fontsize=11)
plt.ylabel('Probabilité de fraude', fontsize=11)
plt.title('🚨 Système d\'alertes par seuils de risque', fontsize=13, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📈 9. Analyse temporelle et géographique des fraudes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraude par heure
taux_heure = df_feat.groupby('heure_soumission')['fraude'].mean()
axes[0].bar(taux_heure.index, taux_heure.values, color='#5C6BC0', edgecolor='white')
axes[0].set_title('Taux de fraude par heure de soumission', fontweight='bold')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Taux de fraude')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].axvspan(-0.5, 7.5, alpha=0.1, color='red', label='Hors heures')
axes[0].axvspan(19.5, 23.5, alpha=0.1, color='red')
axes[0].legend(['Hors heures ouvrées'], fontsize=9)

# Fraude par jour de semaine
jours = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
taux_jour = df_feat.groupby('jour_semaine')['fraude'].mean()
colors_jour = ['#EF5350' if i >= 5 else '#42A5F5' for i in range(7)]
axes[1].bar(jours, taux_jour.values, color=colors_jour, edgecolor='white')
axes[1].set_title('Taux de fraude par jour de la semaine', fontweight='bold')
axes[1].set_ylabel('Taux de fraude')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.suptitle('📅 Analyse temporelle des fraudes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse géographique
geo_analyse = df_feat.groupby('ville').agg(
    nb_notes=('fraude', 'count'),
    taux_fraude=('fraude', 'mean'),
    montant_moyen=('montant', 'mean')
).reset_index().sort_values('taux_fraude', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
colors_geo = ['#F44336' if v in villes_etrangeres else '#2196F3' for v in geo_analyse['ville']]
bars = ax.bar(geo_analyse['ville'], geo_analyse['taux_fraude'],
               color=colors_geo, edgecolor='white', linewidth=1.5)

ax.set_title('Taux de fraude par ville (🔴 étrangère | 🔵 française)', fontsize=13, fontweight='bold')
ax.set_ylabel('Taux de fraude')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Annotations montant moyen
for bar, montant in zip(bars, geo_analyse['montant_moyen']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{montant:.0f}€', ha='center', fontsize=9, color='black')

plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## 📋 10. Synthèse et conclusions

In [ ]:
# Tableau récapitulatif des performances
recap = pd.DataFrame({
    'Modèle': list(resultats.keys()),
    'AUC CV (moy)': [f"{v['auc_cv']:.4f}" for v in resultats.values()],
    'AUC CV (std)': [f"±{v['auc_cv_std']:.4f}" for v in resultats.values()],
    'AUC Test': [f"{v['auc_test']:.4f}" for v in resultats.values()],
})
recap['Meilleur'] = recap['Modèle'] == best_nom
print("🏆 Tableau de comparaison des modèles")
print("="*60)
print(recap.to_string(index=False))

print("\n" + "="*60)
print("📌 CONCLUSIONS")
print("="*60)
print(f"""
1. MODÈLE RECOMMANDÉ : {best_nom}
   → AUC Test : {best_res['auc_test']:.4f}

2. FEATURES LES PLUS PRÉDICTIVES :
   → Absence de justificatifs (sans_justificatif)
   → Montant élevé par rapport au salaire (ratio_montant_salaire)
   → Soumission hors heures ouvrées (est_hors_heures)
   → Voyage à l'étranger (est_etranger)
   → Absence de validation manager (manager_approval = 0)

3. SYSTÈME D'ALERTES :
   → Seuil critique (≥ {SEUIL_CRITIQUE}) : blocage automatique recommandé
   → Seuil élevé   (≥ {SEUIL_ELEVE}) : révision manuelle requise
   → Seuil modéré  (≥ {SEUIL_MODERE}) : surveillance renforcée

4. PISTES D'AMÉLIORATION :
   → Intégrer des données historiques sur plusieurs années
   → Ajouter des features de graphe (réseaux de collègues)
   → Explorer XGBoost / LightGBM pour de meilleures performances
   → Affiner les seuils selon les coûts métier (FP vs FN)
""")